# DuckDB fundamentals for LANTERN

What DuckDB is, how it works, and the concepts Case Study 2 needs for the fact store, `data/facts/facts.duckdb`.

**Before you run**
- Choose the Poetry venv as the kernel (`financial-intelligence-system-...-py3.12`). If VS Code offers to install `ipykernel`, accept.
- Disconnect the VS Code Database Client from `facts.duckdb`. DuckDB locks a file while another program has it open for writing.
- Run the cells top to bottom. Every write goes to an in-memory database; the real `facts.duckdb` is only opened read-only.
- Sections 1–10 work on DuckDB 0.10 and 2.0. Sections 11–12 need 2.0; on an older version those cells print a skip message instead (upgrade with `poetry add duckdb==2.0.0`).
- Clear outputs before committing (see `notebooks/README.md`).

| # | Concept |
|---|---|
| 1 | In-process: no server, in-memory or file |
| 2 | Querying files directly |
| 3 | Results in Python, and querying DataFrames |
| 4 | Tables, types and constraints |
| 5 | Loading data the way `src/facts/build.py` does |
| 6 | JSON columns: citations and lineage |
| 7 | Parameters and scope filters (deal isolation) |
| 8 | Reconciliation: claims vs. facts |
| 9 | The database file: read-only access and locks |
| 10 | Databases, schemas and `ATTACH` |
| 11 | DuckDB 2.0: the `VARIANT` type |
| 12 | DuckDB 2.0: triggers |
| 13 | DuckDB 2.0: agent mode (CLI only) |
| 14 | Gotchas |

## What DuckDB is

- **Embedded.** DuckDB is a library that runs inside your program, like SQLite. There is no server to install, start or connect to.
- **Analytical.** It is built for queries that scan, join and aggregate many rows, not for many small concurrent writes. Data is stored by column, so a query reads only the columns it uses, and it processes rows in batches across all CPU cores.
- **One file, or none.** A database is a single `.duckdb` file, which DVC can track like any other stage output, or it can live only in memory.
- **Reads files and DataFrames directly.** CSV, Parquet, JSON and pandas DataFrames can be queried with SQL without loading them first.

In Case Study 2 it holds the **fact store**: the one place every number the system is allowed to state comes from, each with a tier and a citation.

In [ ]:
import sys
from pathlib import Path

import duckdb
import pandas as pd

# Works whether the notebook runs from notebooks/ or the repo root: find the folder with dvc.yaml
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dvc.yaml").exists())
XBRL_CSV = (ROOT / "reports" / "xbrl_check.csv").as_posix()
FACTS_DB = (ROOT / "data" / "facts" / "facts.duckdb").as_posix()
V2 = int(duckdb.__version__.split(".")[0]) >= 2   # sections 11-12 use DuckDB 2.0 features

print("duckdb", duckdb.__version__)
print("repo  ", ROOT)

## 1. In-process: no server

`duckdb.connect()` with no path gives an **in-memory** database that disappears when the kernel stops. With a path it opens, or creates, a **database file**.

`con.sql(...)` returns a *relation*: a query DuckDB has not run yet. It runs when you ask for results: `.show()` prints a table, `.df()` returns a pandas DataFrame, `.fetchall()` returns a list of tuples. For a single row, use `con.execute(...).fetchone()`; in DuckDB 2.0 relations no longer have `.fetchone()`.

In [ ]:
con = duckdb.connect()   # in-memory: the scratch database for this whole notebook

con.sql("SELECT 42 AS answer, 'hello duck' AS msg").show()

## 2. Query files directly

The file path goes where a table name would. Nothing is loaded or copied. `DESCRIBE` shows the columns and types DuckDB detected; `SUMMARIZE` profiles every column.

`reports/xbrl_check.csv` is the output of the `evaluate` stage: one row per filing line, with the value `parse` read and the SEC's XBRL value.

In [ ]:
con.sql(f"SELECT * FROM '{XBRL_CSV}' LIMIT 5").df()

In [ ]:
con.sql(f"DESCRIBE SELECT * FROM '{XBRL_CSV}'").df()

In [ ]:
con.sql(f"SUMMARIZE SELECT * FROM '{XBRL_CSV}'").df()

In [ ]:
# Grouping and aggregating is what DuckDB is built for
con.sql(f"""
    SELECT ticker, status, count(*) AS lines
    FROM '{XBRL_CSV}'
    GROUP BY ALL
    ORDER BY ticker, status
""").df()

## 3. Results in Python, and querying DataFrames

Results come back in the shape you need. DuckDB can also query a pandas DataFrame by its variable name, so pandas and SQL mix freely.

In [ ]:
query = f"SELECT ticker, period_end, xbrl FROM '{XBRL_CSV}' WHERE line = 'Revenues'"

print(con.sql(query).fetchall())   # a list of tuples
con.sql(query).df()                # a pandas DataFrame

In [ ]:
# A plain pandas DataFrame (dtype=object: DuckDB 0.10 can't read pandas 3's new string dtype)...
sectors = pd.DataFrame({"ticker": ["AAPL", "NFLX"],
                        "sector": ["Information Technology", "Communication Services"]}, dtype=object)

# ...queried by its variable name and joined to the CSV
con.sql(f"""
    SELECT s.sector, x.ticker, x.period_end, x.xbrl AS revenue
    FROM '{XBRL_CSV}' x JOIN sectors s USING (ticker)
    WHERE x.line = 'Revenues'
    ORDER BY x.ticker, x.period_end
""").df()

## 4. Tables, types and constraints

A table has typed columns. **Constraints** make the database enforce rules itself, so a bad row fails when it is inserted instead of slipping through. Case Study 2 needs two such rules: every fact has a tier and a citation, and tier-4/5 values (management and web claims) never enter `facts`.

The column `"end"` is quoted because `END` is a SQL keyword.

In [ ]:
con.execute("""
    CREATE TABLE demo_facts (
        fact_id  VARCHAR PRIMARY KEY,
        metric   VARCHAR,
        value    DOUBLE,
        "end"    DATE,
        tier     TINYINT NOT NULL CHECK (tier BETWEEN 1 AND 3),
        citation JSON NOT NULL
    )
""")
con.execute("""INSERT INTO demo_facts VALUES ('f1', 'revenue', 100.0, '2025-12-31', 1, '{"kind": "ixbrl_context"}')""")
con.sql("SELECT * FROM demo_facts").df()

In [ ]:
# A tier-4 value (a CIM claim) is rejected by the CHECK constraint...
try:
    con.execute("""INSERT INTO demo_facts VALUES ('f2', 'revenue', 120.0, '2025-12-31', 4, '{"kind": "pdf_bbox"}')""")
except duckdb.ConstraintException as e:
    print("rejected:", e)

# ...and so is a fact with no citation
try:
    con.execute("INSERT INTO demo_facts VALUES ('f3', 'revenue', 100.0, '2025-12-31', 1, NULL)")
except duckdb.ConstraintException as e:
    print("rejected:", e)

## 5. Load data the way `src/facts/build.py` does

The pipeline's builder is plain SQL: create the tables, then one `INSERT ... SELECT` that reads the CSV and reshapes each row into a fact. This cell imports that exact SQL and runs it against the in-memory database, so you are looking at the pipeline's logic, not a copy of it.

`INSERT INTO facts BY NAME` matches columns by name, so the `SELECT` can list them in any order and leave out the ones that stay empty.

In [ ]:
sys.path.insert(0, str(ROOT / "src"))
from facts.build import LOAD_XBRL, SCHEMA   # the SQL that src/facts/build.py runs

con.execute(SCHEMA)                          # creates `facts` and `claims`
con.execute(LOAD_XBRL.format(csv=XBRL_CSV))  # loads the 16 tier-1 XBRL facts
con.sql('SELECT entity_id, metric, "end", value, unit, tier FROM facts ORDER BY ALL').df()

In [ ]:
print(LOAD_XBRL)   # read it: each line turns one CSV column into one fact column

## 6. JSON columns: citations and lineage

`citation` and `lineage` are JSON because their contents depend on the source: an XBRL fact cites a concept and a filing, a data-room fact cites a sheet and cell, a CIM claim cites a page and a bounding box. `->>` extracts a field as text; `->` keeps it as JSON.

In [ ]:
con.sql("""
    SELECT fact_id,
           citation ->> 'kind'    AS kind,
           citation ->> 'doc_id'  AS filing,
           citation ->> 'concept' AS concept,
           lineage  ->  'parents' AS parents
    FROM facts
    LIMIT 4
""").df()

## 7. Parameters and scope filters (deal isolation)

Pass values with `?` placeholders, never by pasting them into the SQL string. DuckDB treats them strictly as values, so a stray quote in the input cannot change the query.

The scope filter is the core of deal isolation: a `deal_a` query sees Deal A's records plus public ones, and nothing from Deal B. The next cell adds one made-up Deal B fact and checks that a Deal A query cannot see it.

In [ ]:
# One made-up tier-3 fact from Deal B's data room (seller financials, cited by sheet and cell)
con.execute("""
    INSERT INTO facts (fact_id, deal_id, entity_id, metric, value, unit, "end", tier, source_type, doc_id, citation)
    VALUES ('demo-b-1', 'deal_b', 'TARGET_B', 'revenue', 80000000, 'USD', '2025-12-31', 3, 'dataroom',
            'DR-deal_b-001', json_object('kind', 'sheet_cell', 'locator', 'P&L!C7'))
""")

SCOPES = {"deal_a", "deal_b", "kestrel", "public"}


def facts_in_scope(scope: str, metric: str) -> pd.DataFrame:
    if scope not in SCOPES:
        raise ValueError(f"unknown scope: {scope!r}")   # CS2: a query without a valid scope is refused
    return con.execute(
        """SELECT deal_id, entity_id, metric, "end", value FROM facts
           WHERE metric = ? AND deal_id IN (?, 'public') ORDER BY ALL""",
        [metric, scope],
    ).df()


facts_in_scope("deal_a", "revenue")   # public facts only: Deal B's row is invisible

In [ ]:
facts_in_scope("deal_b", "revenue")   # Deal B sees its own fact plus the public ones

In [ ]:
# The CS2 isolation test in miniature: a deal_a query must return zero Deal B rows
assert (facts_in_scope("deal_a", "revenue")["deal_id"] != "deal_b").all()
print("isolation holds")

## 8. Reconciliation: claims vs. facts

The `reconcile` stage compares each management claim (tier 4) with the seller's financials (tier 3) and labels it. It is a `LEFT JOIN`: every claim is kept, and a claim with no matching fact becomes `no_fact`.

gap = (claim − fact) / |fact|. With the Case Study 2 defaults (`reconcile.tolerance_rel: 0.01`, `reconcile.high_gap_rel: 0.05`), a gap of up to 1% is a match and a gap above 5% is high severity. **All numbers below are made up.**

In [ ]:
# Made-up seller financials for Deal A (tier 3)...
con.execute("""
    INSERT INTO facts (fact_id, deal_id, entity_id, metric, value, unit, "end", tier, source_type, citation)
    VALUES
      ('demo-a-rev',  'deal_a', 'TARGET_A', 'revenue',             50000000, 'USD',   '2025-12-31', 3, 'dataroom', json_object('kind', 'sheet_cell', 'locator', 'P&L!C5')),
      ('demo-a-ebit', 'deal_a', 'TARGET_A', 'ebitda',              10800000, 'USD',   '2025-12-31', 3, 'dataroom', json_object('kind', 'sheet_cell', 'locator', 'P&L!C19')),
      ('demo-a-top1', 'deal_a', 'TARGET_A', 'top1_customer_share',    0.125, 'ratio', '2025-12-31', 3, 'dataroom', json_object('kind', 'sheet_cell', 'locator', 'Customers!D3'))
""")

# ...and made-up claims from the CIM (tier 4), each cited by page and bounding box
con.execute("""
    INSERT INTO claims (claim_id, deal_id, entity_id, metric, value, unit, qualifier, "end", tier, source_type, doc_id, citation)
    VALUES
      ('c1', 'deal_a', 'TARGET_A', 'revenue',             50200000, 'USD',   'FY',       '2025-12-31', 4, 'dataroom', 'DR-deal_a-001', json_object('kind', 'pdf_bbox', 'page', 12, 'bbox', [72, 300, 540, 318])),
      ('c2', 'deal_a', 'TARGET_A', 'ebitda',              12000000, 'USD',   'FY',       '2025-12-31', 4, 'dataroom', 'DR-deal_a-001', json_object('kind', 'pdf_bbox', 'page', 13, 'bbox', [72, 410, 540, 428])),
      ('c3', 'deal_a', 'TARGET_A', 'top1_customer_share',     0.12, 'ratio', 'FY',       '2025-12-31', 4, 'dataroom', 'DR-deal_a-002', json_object('kind', 'pdf_bbox', 'page', 4,  'bbox', [72, 150, 540, 168])),
      ('c4', 'deal_a', 'TARGET_A', 'arr',                 61000000, 'USD',   'run-rate', '2025-12-31', 4, 'dataroom', 'DR-deal_a-003', json_object('kind', 'pdf_bbox', 'page', 2,  'bbox', [72, 500, 540, 518]))
""")

In [ ]:
TOLERANCE, HIGH_GAP = 0.01, 0.05   # CS2 defaults: reconcile.tolerance_rel, reconcile.high_gap_rel

con.execute("""
    WITH joined AS (
        SELECT c.claim_id, c.metric, c.value AS claim, f.fact_id, f.value AS fact,
               (c.value - f.value) / abs(f.value) AS gap
        FROM claims c
        LEFT JOIN facts f
          ON f.deal_id = c.deal_id AND f.metric = c.metric AND f."end" = c."end" AND f.tier = 3
        WHERE c.deal_id = ?
    )
    SELECT claim_id, metric, claim, fact, fact_id, round(gap, 4) AS gap,
           CASE WHEN fact_id IS NULL THEN 'no_fact'
                WHEN abs(gap) <= ?   THEN 'match'
                ELSE 'discrepancy' END AS status,
           CASE WHEN fact_id IS NULL OR abs(gap) <= ? THEN NULL
                WHEN abs(gap) > ?    THEN 'high'
                ELSE 'medium' END AS severity
    FROM joined
    ORDER BY claim_id
""", ["deal_a", TOLERANCE, TOLERANCE, HIGH_GAP]).df()

The real stage also maps metric names, aligns periods, and adds the `definition_mismatch` and `period_mismatch` statuses (for example adjusted vs. reported EBITDA, or fiscal vs. calendar years). The join above is its core.

## 9. The database file: read-only access and locks

Everything so far lived in memory. The pipeline's output is a file, `data/facts/facts.duckdb`, built by `python src/facts/build.py`. Programs that only read it should open it with `read_only=True`: several readers can share a file, but a writer locks everyone else out. If you get a lock error, close whatever else has the file open (the VS Code extension, the CLI, another notebook).

In [ ]:
try:
    with duckdb.connect(FACTS_DB, read_only=True) as db:
        print(db.sql("SHOW TABLES").fetchall())
        print(db.execute("SELECT count(*) AS facts, count(DISTINCT entity_id) AS companies FROM facts").fetchone())
        try:
            db.execute("CREATE TABLE scratch (x INTEGER)")   # writing is refused in read-only mode
        except duckdb.Error as e:
            print("write refused:", str(e).splitlines()[0])
except duckdb.IOException as e:
    print("Could not open the file. Build it with `python src/facts/build.py`, "
          "and close any other program that has it open.\n", e)

## 10. Databases, schemas and `ATTACH`

One connection can see several databases at once, and every table has a three-part name: `database.schema.table`. A database is named after its file (`facts.duckdb` becomes `facts`) or after the alias you give `ATTACH`, and its default schema is `main`. A bare table name is looked up in the **current** database.

This is why the VS Code extension said `Table with name facts does not exist`: its current database was an empty in-memory one, and the file was attached next to it.

In [ ]:
con.execute(f"ATTACH '{FACTS_DB}' AS lantern (READ_ONLY)")
con.sql("SELECT database_name, path, readonly FROM duckdb_databases() WHERE NOT internal").df()

In [ ]:
print(con.execute("SELECT current_database()").fetchone())               # memory
print(con.execute("SELECT count(*) FROM facts").fetchone())               # memory.main.facts: this notebook's table
print(con.execute("SELECT count(*) FROM lantern.main.facts").fetchone())  # the file's table

In [ ]:
con.execute("USE lantern")       # switch the current database...
try:
    print(con.execute("SELECT current_database(), count(*) FROM facts").fetchone())   # now a bare `facts` means the file
finally:
    con.execute("USE memory")    # ...and always switch back, even if the query fails
con.execute("DETACH lantern")

## 11. DuckDB 2.0: the `VARIANT` type

A `VARIANT` column holds values of different shapes: each row can have different fields. DuckDB 2.0 detects the common structure and stores it column by column ("shredding"), so queries stay fast. That suits raw extraction output from the CIM, deck and transcript, where each claim carries different details. Fields are read with dot notation, and a missing field is `NULL`.

In [ ]:
if not V2:
    print(f"Skipped: VARIANT needs DuckDB 2.0, this kernel has {duckdb.__version__}")
else:
    con.execute("CREATE TABLE claims_raw (doc_id VARCHAR, payload VARIANT)")
    con.execute("""
        INSERT INTO claims_raw VALUES
          ('DR-deal_a-001', '{"metric": "revenue", "value": 52.3, "unit": "USD_m", "qualifier": "LTM"}'::JSON::VARIANT),
          ('DR-deal_a-002', '{"metric": "top1_customer_share", "value": 0.18, "page": 14, "bbox": [72, 410, 540, 432]}'::JSON::VARIANT),
          ('DR-deal_a-003', '{"metric": "arr", "value": 61.0, "qualifier": "run-rate", "speaker": "CFO"}'::JSON::VARIANT)
    """)
    display(con.sql("SELECT doc_id, variant_type(payload) AS type, variant_keys(payload) AS keys FROM claims_raw").df())

In [ ]:
# Cast fields to ordinary types to filter, join or do arithmetic with them
if not V2:
    print(f"Skipped: VARIANT needs DuckDB 2.0, this kernel has {duckdb.__version__}")
else:
    display(con.sql("""
        SELECT doc_id,
               payload.metric::VARCHAR    AS metric,
               payload.value::DOUBLE      AS value,
               payload.qualifier::VARCHAR AS qualifier
        FROM claims_raw
    """).df())

## 12. DuckDB 2.0: triggers

A trigger is SQL that runs automatically when a table changes. The 2.0 announcement shows two uses: an **audit log** that records what changed, and a **summary that keeps itself up to date**. In DuckDB 2.0, write them `FOR EACH STATEMENT` with `REFERENCING OLD TABLE / NEW TABLE`: the changed rows appear as small tables inside the trigger.

In [ ]:
# Audit log: every UPDATE to a fact's value is recorded with the old and new value
if not V2:
    print(f"Skipped: triggers need DuckDB 2.0, this kernel has {duckdb.__version__}")
else:
    con.execute("CREATE TABLE facts_audit (fact_id VARCHAR, old_value DOUBLE, new_value DOUBLE, changed_at TIMESTAMP)")
    con.execute("""
        CREATE TRIGGER trg_facts_audit AFTER UPDATE ON facts
        REFERENCING OLD TABLE AS old_rows NEW TABLE AS new_rows
        FOR EACH STATEMENT
        INSERT INTO facts_audit
        SELECT o.fact_id, o.value, n.value, now() FROM old_rows o JOIN new_rows n USING (fact_id)
    """)

    con.execute("UPDATE facts SET value = 51000000 WHERE fact_id = 'demo-a-rev'")   # the seller restates revenue
    display(con.sql("SELECT fact_id, old_value, new_value FROM facts_audit").df())

In [ ]:
# Self-updating summary: facts per deal, adjusted on every insert instead of recounted
if not V2:
    print(f"Skipped: triggers need DuckDB 2.0, this kernel has {duckdb.__version__}")
else:
    con.execute("CREATE TABLE facts_per_deal AS SELECT deal_id, count(*) AS n_facts FROM facts GROUP BY deal_id")
    con.execute("""
        CREATE TRIGGER trg_facts_per_deal AFTER INSERT ON facts
        REFERENCING NEW TABLE AS n
        FOR EACH STATEMENT
        MERGE INTO facts_per_deal s
        USING (SELECT deal_id, count(*) AS k FROM n GROUP BY deal_id) d ON s.deal_id = d.deal_id
        WHEN MATCHED THEN UPDATE SET n_facts = s.n_facts + d.k
        WHEN NOT MATCHED THEN INSERT VALUES (d.deal_id, d.k)
    """)

    con.execute("""
        INSERT INTO facts (fact_id, deal_id, entity_id, metric, value, unit, "end", tier, source_type, citation)
        VALUES ('demo-k-1', 'kestrel', 'KESTREL', 'revenue', 30000000, 'USD', '2025-12-31', 3, 'dataroom',
                json_object('kind', 'sheet_cell', 'locator', 'P&L!C5'))
    """)
    display(con.sql("SELECT * FROM facts_per_deal ORDER BY deal_id").df())   # kestrel appears without a recount

## 13. DuckDB 2.0: agent mode (CLI only)

When an AI coding agent drives the DuckDB **command-line tool**, 2.0 switches to output built for it: compact Markdown tables with the types in the header, errors as JSON, and large results cut to their first and last rows with a notice and a hash. It turns on by itself when an agent's environment variable is set; `-agent` and `-no-agent` force it on or off. It does not apply to Python.

To see it, run this in a terminal from the repo root with a DuckDB 2.0 CLI, with the VS Code extension disconnected:

```bash
duckdb -agent -readonly data/facts/facts.duckdb -c "SELECT entity_id, value FROM facts WHERE metric = 'revenue'"
```

Agent mode prints:

```
| entity_id:VARCHAR | value:DOUBLE |
|---|---|
| AAPL | 391035000000.0 |
...
```

Run it again with `-no-agent` to compare with the usual boxed table.

## 14. Gotchas

- **Quote `"end"`.** `END` is a SQL keyword.
- **Versions.** A file written by DuckDB 2.0 cannot be opened by older versions. Keep Poetry, the CLI and the VS Code extension on 2.0.x.
- **One writer at a time.** Open files with `read_only=True` unless you are the stage that builds them, and close connections when you are done.
- **`?` for values.** Use f-strings only for trusted things such as file paths.
- **One row in 2.0.** `con.sql(...).fetchone()` no longer exists; use `con.execute(...).fetchone()`.
- **Bare names use the current database.** With several databases attached, write `database.schema.table` or switch with `USE`.
- **Git hygiene.** `facts.duckdb` belongs in DVC, never in Git. Clear notebook outputs before committing.

In [ ]:
con.close()